In [0]:
%python
# 1. Variables to authenticate connection
container_name = "omnichannel-customer-insights"
storage_account_name = "saaurawellnessssc"
storage_account_key = "OwiEKdkkGiKkBi2IU+dwPLDuSpIrnDlL1mIrwPWj7i0tMfCIZEpcLwLpG7ZW94l/hiAkN7rOnyTK+AStQBEVqg==" # Replace with your key

# 2. Define the paths to your clean Silver Delta tables
silver_crm_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/crm_customer_profiles/"
silver_web_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/web_clickstream_events/"

# 3. Load the data from your Azure Data Lake
crm = spark.read.format("delta").option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key).load(silver_crm_path)
web = spark.read.format("delta").option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key).load(silver_web_path)

# 4. Register them as temporary SQL views for the guidelines mapping
crm.createOrReplaceTempView("silver_crm")
web.createOrReplaceTempView("silver_web")

print("🖥️ Silver views successfully registered! Ready for SQL analytical transformations.")

🖥️ Silver views successfully registered! Ready for SQL analytical transformations.


In [0]:
%sql
-- Calculate Recency and Frequency metrics grouped by customer
SELECT 
    customer_id,
    
    -- Frequency: Count how many pages they visited in total
    COUNT(page_visited) AS frequency,
    
    -- Recency: Calculate days between their last activity and a fixed baseline date (representing "today")
    DATEDIFF(CAST('2026-06-30' AS DATE), MAX(CAST(timestamp AS DATE))) AS recency
FROM 
    silver_web
GROUP BY 
    customer_id

customer_id,frequency,recency
CUST-51245,7,32
CUST-82340,13,33
CUST-97012,11,29
CUST-16734,8,29
CUST-73956,10,49
CUST-86271,13,43
CUST-16572,9,79
CUST-12875,12,35
CUST-71524,6,41
CUST-60488,12,32


In [0]:
%sql
-- Fusing CRM profiles with behavioral traffic and implementing CLV math
SELECT 
    c.customer_id,
    c.state,
    c.signup_date,
    c.total_historical_spend,
    w.frequency,
    w.recency,
    
    -- The Blueprint CLV Formula: Historical Spend adjusted by annualized transaction velocity
    ROUND(c.total_historical_spend * ((w.frequency / DATEDIFF(CAST('2026-06-30' AS DATE), CAST(c.signup_date AS DATE))) * 365), 2) AS clv
FROM 
    silver_crm c
INNER JOIN (
    SELECT 
        customer_id,
        COUNT(page_visited) AS frequency,
        DATEDIFF(CAST('2026-06-30' AS DATE), MAX(CAST(timestamp AS DATE))) AS recency
    FROM 
        silver_web
    GROUP BY 
        customer_id
) w ON c.customer_id = w.customer_id

customer_id,state,signup_date,total_historical_spend,frequency,recency,clv
CUST-51245,NM,2024-01-20,4148.98,7,32,11884.13
CUST-82340,MS,2025-09-14,1326.64,13,33,21781.68
CUST-97012,MA,2025-04-24,4878.55,11,29,45341.15
CUST-16734,VI,2024-03-03,1581.77,8,29,5440.25
CUST-73956,AZ,2026-01-12,2924.17,10,49,63155.15
CUST-86271,MA,2025-08-14,3450.36,13,43,51162.37
CUST-16572,NY,2025-03-02,2608.85,9,79,17670.25
CUST-12875,TN,2026-05-07,342.96,12,35,27817.87
CUST-71524,MA,2024-11-06,4064.47,6,41,14810.63
CUST-60488,ID,2026-02-10,1160.01,12,32,36291.74


In [0]:
%sql
-- Creating our final marketing segments based on behavior and value rules
SELECT 
    customer_id,
    state,
    total_historical_spend,
    frequency,
    recency,
    clv,
    CASE 
        -- Rule 1: High historical value and interacted recently
        WHEN total_historical_spend > 1000 AND recency <= 30 THEN 'VIP Champions'
        
        -- Rule 2: Interacted months ago and spent a decent amount
        WHEN total_historical_spend > 200 AND recency > 120 THEN 'At-Risk Churn'
        
        -- Rule 3: Catch-all for regular active users
        ELSE 'Standard Active'
    END AS marketing_segment
FROM (
    SELECT 
        c.customer_id,
        c.state,
        c.signup_date,
        c.total_historical_spend,
        w.frequency,
        w.recency,
        ROUND(c.total_historical_spend * ((w.frequency / DATEDIFF(CAST('2026-06-30' AS DATE), CAST(c.signup_date AS DATE))) * 365), 2) AS clv
    FROM 
        silver_crm c
    INNER JOIN (
        SELECT 
            customer_id,
            COUNT(page_visited) AS frequency,
            DATEDIFF(CAST('2026-06-30' AS DATE), MAX(CAST(timestamp AS DATE))) AS recency
        FROM 
            silver_web
        GROUP BY 
            customer_id
    ) w ON c.customer_id = w.customer_id
)

customer_id,state,total_historical_spend,frequency,recency,clv,marketing_segment
CUST-51245,NM,4148.98,7,32,11884.13,Standard Active
CUST-82340,MS,1326.64,13,33,21781.68,Standard Active
CUST-97012,MA,4878.55,11,29,45341.15,VIP Champions
CUST-16734,VI,1581.77,8,29,5440.25,VIP Champions
CUST-73956,AZ,2924.17,10,49,63155.15,Standard Active
CUST-86271,MA,3450.36,13,43,51162.37,Standard Active
CUST-16572,NY,2608.85,9,79,17670.25,Standard Active
CUST-12875,TN,342.96,12,35,27817.87,Standard Active
CUST-71524,MA,4064.47,6,41,14810.63,Standard Active
CUST-60488,ID,1160.01,12,32,36291.74,Standard Active


In [0]:
%python
# 1. Capture the entire SQL query output into a Spark DataFrame
df_gold_final = spark.sql("""
SELECT 
    customer_id,
    state,
    total_historical_spend,
    frequency,
    recency,
    clv,
    CASE 
        WHEN total_historical_spend > 1000 AND recency <= 30 THEN 'VIP Champions'
        WHEN total_historical_spend > 200 AND recency > 120 THEN 'At-Risk Churn'
        ELSE 'Standard Active'
    END AS marketing_segment
FROM (
    SELECT 
        c.customer_id,
        c.state,
        c.signup_date,
        c.total_historical_spend,
        w.frequency,
        w.recency,
        ROUND(c.total_historical_spend * ((w.frequency / DATEDIFF(CAST('2026-06-30' AS DATE), CAST(c.signup_date AS DATE))) * 365), 2) AS clv
    FROM 
        silver_crm c
    INNER JOIN (
        SELECT 
            customer_id,
            COUNT(page_visited) AS frequency,
            DATEDIFF(CAST('2026-06-30' AS DATE), MAX(CAST(timestamp AS DATE))) AS recency
        FROM 
            silver_web
        GROUP BY 
            customer_id
    ) w ON c.customer_id = w.customer_id
)
""")

# 2. Define your exact target Gold layer path on Azure
gold_save_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold/customer_marketing_insights/"

# 3. Permanently write it down as an enterprise Delta table
df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key) \
    .save(gold_save_path)

print("🏆 MASTER PIECE SECURED! Your finalized analytical Gold table has been saved to Azure!")

🏆 MASTER PIECE SECURED! Your finalized analytical Gold table has been saved to Azure!
